# 03 — Inference & Evaluation

Batch inference across all three modes, full metrics comparison table (accuracy, NLL, Brier, ECE),
per-mode uncertainty columns, and calibration curves.

> **Note:** This notebook uses the pretrained `compliance-v1` model. To use your custom-trained model,
> replace the paths at the top of each cell with your `user_workspace/checkpoints/` paths.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'epalea', '-q'])
import epalea
print(f'epalea {epalea.__version__} ready')

## Load the pretrained model

In [ ]:
import epalea

model = epalea.load_model("compliance-v1")
print(f"Available modes: {model.available_modes}")

## Single inference — all three modes

In [ ]:
import subprocess, json

def parse_cli_output(stdout):
    """Strip non-JSON prefix (HF warnings etc) and parse."""
    return json.loads(stdout[stdout.index('{'):])

# Single inference
result = subprocess.run([
    "epalea", "infer",
    "--checkpoint", "./user_workspace/checkpoints/compliance/best_model.pt",
    "--aggregator-checkpoint", "./user_workspace/checkpoints/compliance/aggregator.pt",
    "--schema", "./user_workspace/checkpoints/compliance/schema.json",
    "--index-dir", "./user_workspace/data/compliance",
    "--entity-id", "C0001",
    "--mode", "both",
], capture_output=True, text=True)

output = parse_cli_output(result.stdout)
print(f"SPN:  {output['results']['spn']['prediction']}  conf={output['results']['spn']['confidence']:.3f}")
print(f"  epistemic={output['uncertainty']['spn']['epistemic']}  aleatoric={output['uncertainty']['spn']['aleatoric']}")
print(f"Agg:  {output['results']['aggregator']['prediction']}  conf={output['results']['aggregator']['confidence']:.3f}")
print(f"  epistemic={output['uncertainty']['aggregator']['epistemic']}  aleatoric={output['uncertainty']['aggregator']['aleatoric']}")

## Batch inference

In [ ]:
# Batch inference
result = subprocess.run([
    "epalea", "batch-infer",
    "--checkpoint", "./user_workspace/checkpoints/compliance/best_model.pt",
    "--aggregator-checkpoint", "./user_workspace/checkpoints/compliance/aggregator.pt",
    "--schema", "./user_workspace/checkpoints/compliance/schema.json",
    "--index-dir", "./user_workspace/data/compliance",
    "--companies", "./user_workspace/data/compliance/test_companies.json",
    "--mode", "both",
    "--output-dir", "./user_workspace/results",
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

with open("./user_workspace/results/batch_predictions.json") as f:
    batch_results = json.load(f)

print(f"Batch complete: {len(batch_results)} entities")
sample = batch_results[5]
print(f"Sample: {sample['entity_id']} → {sample['results']['spn']['prediction']}")
print(f"  uncertainty.spn:        epistemic={sample['uncertainty']['spn']['epistemic']}")
print(f"  uncertainty.aggregator: epistemic={sample['uncertainty']['aggregator']['epistemic']}")

## Evaluate

In [ ]:
# Evaluation
result = subprocess.run([
    "epalea", "evaluate",
    "--checkpoint", "./user_workspace/checkpoints/compliance/best_model.pt",
    "--aggregator-checkpoint", "./user_workspace/checkpoints/compliance/aggregator.pt",
    "--schema", "./user_workspace/checkpoints/compliance/schema.json",
    "--index-dir", "./user_workspace/data/compliance",
    "--test-companies", "./user_workspace/data/compliance/test_companies.json",
    "--mode", "both",
    "--output-dir", "./user_workspace/results",
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])